In [1]:
import sys; sys.path.append('..')
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/wallets/transaction_dataset.csv")
print(df.shape)  # avant nettoyage

df.columns = df.columns.str.strip()          # enlève les espaces parasites
df = df.drop(columns=["Unnamed: 0", "Index"], errors="ignore")
df = df.drop_duplicates(subset="Address")

y = df["FLAG"]
addresses = df["Address"]
X = df.drop(columns=["FLAG", "Address"]).select_dtypes("number")
X = X.loc[:, X.nunique(dropna=False) > 1]

print(X.shape)
print(y.value_counts(normalize=True))
print(list(X.columns))

(9841, 51)
(9816, 45)
FLAG
0    0.778015
1    0.221985
Name: proportion, dtype: float64
['Avg min between sent tnx', 'Avg min between received tnx', 'Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx', 'Number of Created Contracts', 'Unique Received From Addresses', 'Unique Sent To Addresses', 'min value received', 'max value received', 'avg val received', 'min val sent', 'max val sent', 'avg val sent', 'min value sent to contract', 'max val sent to contract', 'avg value sent to contract', 'total transactions (including tnx to create contract', 'total Ether sent', 'total ether received', 'total ether sent contracts', 'total ether balance', 'Total ERC20 tnxs', 'ERC20 total Ether received', 'ERC20 total ether sent', 'ERC20 total Ether sent contract', 'ERC20 uniq sent addr', 'ERC20 uniq rec addr', 'ERC20 uniq sent addr.1', 'ERC20 uniq rec contract addr', 'ERC20 avg time between sent tnx', 'ERC20 avg time between rec tnx', 'ERC20 avg time between rec 2 tnx', 'ERC20 avg ti

In [3]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

addresses.loc[X_te.index].to_csv("../data/processed/wallet_test_addresses.csv", index=False)

print(X_tr.shape, X_te.shape)
print(y_tr.mean(), y_te.mean())

(7852, 45) (1964, 45)
0.22198166072338257 0.2219959266802444


In [4]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

models = {
    "xgb": XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8,
                         colsample_bytree=0.8, eval_metric="logloss", random_state=42),
    "lgbm": LGBMClassifier(n_estimators=400, learning_rate=0.05, random_state=42, verbose=-1),
    "rf": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
}

def make_pipe(clf):
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("smote", SMOTE(random_state=42)),
                     ("clf", clf)])

In [5]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(5, shuffle=True, random_state=42)

for name, clf in models.items():
    s = cross_validate(make_pipe(clf), X_tr, y_tr, cv=cv,
                       scoring=["recall", "precision", "f1", "average_precision"])
    print(name, {k[5:]: round(v.mean(), 3) for k, v in s.items() if k.startswith("test_")})

xgb {'recall': np.float64(0.97), 'precision': np.float64(0.972), 'f1': np.float64(0.971), 'average_precision': np.float64(0.995)}
lgbm {'recall': np.float64(0.971), 'precision': np.float64(0.97), 'f1': np.float64(0.97), 'average_precision': np.float64(0.996)}
rf {'recall': np.float64(0.96), 'precision': np.float64(0.958), 'f1': np.float64(0.959), 'average_precision': np.float64(0.993)}


In [6]:
import json
cv_results = {}
for name, clf in models.items():
    s = cross_validate(make_pipe(clf), X_tr, y_tr, cv=cv,
                       scoring=["recall", "precision", "f1", "average_precision"])
    cv_results[name] = {k[5:]: float(v.mean()) for k, v in s.items() if k.startswith("test_")}

with open("../results/m3_cv_comparison.json", "w") as f:
    json.dump(cv_results, f, indent=2)
cv_results

{'xgb': {'recall': 0.9695962190824359,
  'precision': 0.9718426919946598,
  'f1': 0.9707058576410443,
  'average_precision': 0.994674313832624},
 'lgbm': {'recall': 0.9707407041464942,
  'precision': 0.9702194016140501,
  'f1': 0.9704716842372886,
  'average_precision': 0.9958022554732537},
 'rf': {'recall': 0.9598359845865033,
  'precision': 0.9582299563009267,
  'f1': 0.9590112513811437,
  'average_precision': 0.9930777614416135}}

In [7]:
from sklearn.feature_selection import mutual_info_classif

mi = pd.Series(
    mutual_info_classif(X_tr.fillna(X_tr.median()), y_tr, random_state=42),
    index=X_tr.columns).sort_values(ascending=False)

top20 = mi.head(20).index.tolist()
mi.to_csv("../results/m3_info_gain.csv", header=["info_gain"])

print(mi.head(20))
print("Features ERC20 dans le top 20 :", sum("ERC20" in c for c in top20))

Total ERC20 tnxs                                        0.282750
ERC20 uniq rec addr                                     0.255959
ERC20 uniq rec token name                               0.250518
ERC20 uniq rec contract addr                            0.245147
Time Diff between first and last (Mins)                 0.237339
total ether balance                                     0.189939
max value received                                      0.182622
avg val received                                        0.171854
total ether received                                    0.166402
ERC20 min val rec                                       0.147958
ERC20 avg val rec                                       0.140637
total transactions (including tnx to create contract    0.123457
Unique Received From Addresses                          0.117477
Received Tnx                                            0.113212
min value received                                      0.103383
Avg min between sent tnx 

In [8]:
import joblib
from src.common.metrics import evaluate

for tag, cols in [("all", list(X.columns)), ("ig20", top20)]:
    pipe = make_pipe(models["xgb"]).fit(X_tr[cols], y_tr)
    p = pipe.predict_proba(X_te[cols])[:, 1]
    evaluate(y_te, (p >= 0.5).astype(int), p, f"m3_xgb_{tag}", out_dir="../results")

final = make_pipe(models["xgb"]).fit(X_tr[top20], y_tr)
joblib.dump({"pipeline": final, "features": top20}, "../models/wallet-xgb-v1.joblib")

{'recall': 0.952, 'precision': 0.972, 'f1': 0.962, 'pr_auc': 0.994, 'roc_auc': 0.998}
{'recall': 0.959, 'precision': 0.968, 'f1': 0.963, 'pr_auc': 0.993, 'roc_auc': 0.998}


['../models/wallet-xgb-v1.joblib']

In [9]:
with open("../results/m3_wallet_columns.json", "w") as f:
    json.dump(list(X.columns), f, indent=2)

In [10]:
LIVE_COLS = [
    "Avg min between sent tnx", "Avg min between received tnx",
    "Time Diff between first and last (Mins)", "Sent tnx", "Received Tnx",
    "Unique Received From Addresses", "Unique Sent To Addresses",
    "min value received", "max value received", "avg val received",
    "min val sent", "max val sent", "avg val sent",
    "total Ether sent", "total ether received", "total ether balance",
]
assert all(c in X.columns for c in LIVE_COLS)   # vérifie les noms

live = make_pipe(models["xgb"]).fit(X_tr[LIVE_COLS], y_tr)
p = live.predict_proba(X_te[LIVE_COLS])[:, 1]
evaluate(y_te, (p >= 0.5).astype(int), p, "m3_xgb_live", out_dir="../results")

joblib.dump({"pipeline": live, "features": LIVE_COLS}, "../models/wallet-xgb-live-v1.joblib")

{'recall': 0.858, 'precision': 0.874, 'f1': 0.866, 'pr_auc': 0.959, 'roc_auc': 0.985}


['../models/wallet-xgb-live-v1.joblib']